In [1]:
import psycopg2
from faker import Faker
import random

# Inicializa o Faker para o Brasil
fake = Faker('pt_BR')

# --- CONFIGURAÇÃO DAS CREDENCIAIS ---
DB_CONFIG = {
    "host": "db",
    "database": "datawarehouse",
    "user": "postgres",
    "password": "password123", 
    "port": "5432"
}

In [3]:
def generate_realistic_data():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        cur.execute("SET search_path TO ecommerce, public;")

        # 1. ETNIAS
        etnias = ['Parda', 'Branca', 'Preta', 'Amarela', 'Indígena']
        etnia_ids = []
        for e in etnias:
            cur.execute("INSERT INTO etnia (descricao) VALUES (%s) RETURNING id_etnia", (e,))
            etnia_ids.append(cur.fetchone()[0])

        # 2. CATEGORIAS (Fixo)
        categorias_dados = [
            ('Camisetas', 'Camisas de algodão e casuais'),
            ('Calças', 'Jeans e sarja'),
            ('Vestidos', 'Vestidos de festa e casuais'),
            ('Moda Praia', 'Biquínis e sungas'),
            ('Acessórios', 'Bonés, cintos e bolsas')
        ]
        cat_map = {} # Para mapear Nome -> ID
        for nome, desc in categorias_dados:
            cur.execute("INSERT INTO categoria (nome, descricao) VALUES (%s, %s) RETURNING id_categoria", (nome, desc))
            cat_map[nome] = cur.fetchone()[0]

        # 3. PRODUTOS (Fixo e Relacionado)
        produtos_especificos = [
            ('Camisetas', 'Camiseta Oversized Preta', 'Algodão 30.1 penteado', 89.90),
            ('Camisetas', 'Baby Look Branca Basic', 'Corte feminino ajustado', 59.90),
            ('Calças', 'Calça Jeans Slouchy', 'Jeans lavagem clara', 189.90),
            ('Calças', 'Calça Sarja Bege', 'Modelagem chino casual', 159.90),
            ('Vestidos', 'Vestido Midi Floral', 'Estampa primavera verão', 249.90),
            ('Vestidos', 'Vestido Curto Satin', 'Tecido acetinado festa', 199.90),
            ('Moda Praia', 'Biquíni Cortininha Classic', 'Proteção UV 50+', 120.00),
            ('Moda Praia', 'Sunga Boxer Sport', 'Secagem rápida lycra', 85.00),
            ('Acessórios', 'Cinto Couro Legítimo', 'Fivela metálica grafite', 110.00),
            ('Acessórios', 'Boné Aba Curva Street', 'Fechamento strapback', 75.00)
        ]
        prod_ids = []
        for cat_nome, nome, desc, preco in produtos_especificos:
            cur.execute("INSERT INTO produto (id_categoria, nome, descricao, preco_base) VALUES (%s, %s, %s, %s) RETURNING id_produto",
                        (cat_map[cat_nome], nome, desc, preco))
            prod_ids.append(cur.fetchone()[0])

        # 4. COMPONENTES (Cenário 5)
        # Cores
        cores = [('Azul Marinho', 5.0), ('Preto Absoluto', 0.0), ('Branco Off-White', 0.0), ('Rosa Pastel', 8.0), ('Cinza Mescla', 4.0)]
        cor_ids = [cur.execute("INSERT INTO cores (nome, valor_adicional, quantidade_estoque) VALUES (%s, %s, %s) RETURNING id_cor", (c[0], c[1], random.randint(30, 100))) or cur.fetchone()[0] for c in cores]
        
        # Materiais
        materiais = [('Algodão Pima', 15.0), ('Linho Orgânico', 20.0), ('Poliéster Reciclado', 0.0), ('Viscose Soft', 5.0), ('Jeans Premium', 10.0)]
        mat_ids = [cur.execute("INSERT INTO materiais (nome, valor_adicional, quantidade_estoque) VALUES (%s, %s, %s) RETURNING id_material", (m[0], m[1], random.randint(20, 80))) or cur.fetchone()[0] for m in materiais]

        # Estampas
        estampas = [('Sem Estampa', 0.0), ('Silk Minimalista', 10.0), ('Bordado Frontal', 25.0), ('Floral Sublimado', 15.0), ('Tie-Dye', 12.0)]
        estp_ids = [cur.execute("INSERT INTO estampas (nome, valor_adicional, quantidade_estoque) VALUES (%s, %s, %s) RETURNING id_estampa", (e[0], e[1], random.randint(10, 50))) or cur.fetchone()[0] for e in estampas]

        # 5. FORNECEDORES (5 Inserts)
        forn_ids = []
        for _ in range(5):
            cur.execute("INSERT INTO fornecedor (nome_fantasia, cnpj, contato) VALUES (%s, %s, %s) RETURNING id_fornecedor", (fake.company(), fake.cnpj(), fake.email()))
            forn_ids.append(cur.fetchone()[0])

        # 6. CLIENTES (5 Inserts com 2 Endereços cada)
        for _ in range(5):
            cur.execute("INSERT INTO cliente (nome, cpf, email, idade, id_etnia) VALUES (%s, %s, %s, %s, %s) RETURNING id_cliente",
                        (fake.name(), fake.cpf(), fake.email(), random.randint(20, 50), random.choice(etnia_ids)))
            cid = cur.fetchone()[0]
            for tipo in ['Entrega', 'Cobrança']:
                cur.execute("INSERT INTO endereco (id_cliente, tipo_endereco, cep, logradouro, bairro, cidade) VALUES (%s, %s, %s, %s, %s, %s)",
                            (cid, tipo, fake.postcode(), fake.street_name(), 'Bairro Moda', fake.city()))

        # 7. PEDIDOS E ITENS (Cenário 2)
        # Cria 5 pedidos vinculando os produtos criados
        for _ in range(5):
            cur.execute("INSERT INTO pedido (id_cliente, id_endereco_entrega, valor_total_pedido) VALUES (%s, %s, %s) RETURNING id_pedido",
                        (random.choice(range(1, 6)), random.choice(range(1, 11)), 0))
            ped_id = cur.fetchone()[0]
            
            # Cada pedido tem 1 item para simplificar o cálculo do preço final
            p_id = random.choice(prod_ids)
            cur.execute("SELECT preco_base FROM produto WHERE id_produto = %s", (p_id,))
            p_base = float(cur.fetchone()[0])
            
            # Soma custo fixo de personalização (Ex: Cor + Material)
            adicional = 20.0 # Valor fictício somado
            total_item = p_base + adicional
            
            cur.execute("INSERT INTO item_pedido (id_pedido, id_produto, quantidade, preco_base_aplicado, preco_personalizacoes_total) VALUES (%s, %s, %s, %s, %s)",
                        (ped_id, p_id, 1, p_base, adicional))
            
            cur.execute("UPDATE pedido SET valor_total_pedido = %s WHERE id_pedido = %s", (total_item, ped_id))

        conn.commit()
        print("Sucesso! Dados de moda inseridos e relacionados.")

    except Exception as e:
        print(f"Erro: {e}")
        conn.rollback()
    finally:
        cur.close()
        conn.close()

if __name__ == "__main__":
    generate_realistic_data()

Sucesso! Dados de moda inseridos e relacionados.
